# 05 - Soft Voting Ensemble

Combine the Random Forest and Extra Trees classifiers using probability-based soft voting. Both estimators are complete sklearn pipelines, so preprocessing is fit only on training folds and is retained for backend inference. This notebook compares all candidates on the same held-out split and saves the ensemble for the API.

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'student_performance_processed.csv'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH)
target = 'performance_category'
excluded = {'student_id', target, 'risk_level', 'final_marks', 'passed'}
features = [column for column in df.columns if column not in excluded]
X = df[features].copy()
y = df[target].astype(str)
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Dataset:', df.shape, '| Feature count:', len(features))

In [ ]:
def make_preprocessor():
    return ColumnTransformer([
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ])

def make_pipeline(classifier):
    return Pipeline([('preprocessor', make_preprocessor()), ('classifier', classifier)])

# Train comparable individual baselines and a two-estimator soft-voting ensemble.
candidates = {
    'Logistic Regression': make_pipeline(LogisticRegression(max_iter=1500)),
    'Random Forest': make_pipeline(RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1)),
    'Extra Trees': make_pipeline(ExtraTreesClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1)),
}
ensemble = VotingClassifier(
    estimators=[
        ('random_forest', candidates['Random Forest']),
        ('extra_trees', candidates['Extra Trees']),
    ],
    voting='soft',
    n_jobs=-1,
)
candidates['Random Forest'].fit(X_train, y_train)
candidates['Extra Trees'].fit(X_train, y_train)
candidates['Logistic Regression'].fit(X_train, y_train)
ensemble.fit(X_train, y_train)
candidates['Two-model Soft Voting Ensemble'] = ensemble

In [ ]:
metrics = []
for model_name, fitted_model in candidates.items():
    predictions = fitted_model.predict(X_test)
    row = {
        'model': model_name,
        'accuracy': round(accuracy_score(y_test, predictions), 4),
        'precision_macro': round(precision_score(y_test, predictions, average='macro', zero_division=0), 4),
        'recall_macro': round(recall_score(y_test, predictions, average='macro', zero_division=0), 4),
        'f1_macro': round(f1_score(y_test, predictions, average='macro', zero_division=0), 4),
    }
    metrics.append(row)
comparison = pd.DataFrame(metrics).sort_values('f1_macro', ascending=False)
display(comparison)
best_name = comparison.iloc[0]['model']
print('Best model by macro-F1:', best_name)
ensemble_predictions = ensemble.predict(X_test)
print('Ensemble classification report:\n', classification_report(y_test, ensemble_predictions, zero_division=0))
print('Ensemble confusion matrix:\n', confusion_matrix(y_test, ensemble_predictions))

In [ ]:
artifact_names = {
    'Logistic Regression': 'student_performance_model.pkl',
    'Random Forest': 'student_performance_random_forest.pkl',
    'Extra Trees': 'student_performance_extra_trees.pkl',
    'Two-model Soft Voting Ensemble': 'student_performance_ensemble.pkl',
}
for model_name, filename in artifact_names.items():
    artifact_path = MODEL_DIR / filename
    with artifact_path.open('wb') as model_file:
        pickle.dump(candidates[model_name], model_file)

metrics_path = MODEL_DIR / 'model_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print('Saved model artifacts:')
for filename in artifact_names.values():
    print(f' - {MODEL_DIR / filename}')
print(f'Saved model comparison: {metrics_path}')

## Conclusion

This notebook compares three standalone classifiers with a soft-voting ensemble of Random Forest and Extra Trees. It saves every evaluated estimator and the held-out metrics. The API selects the available artifact with the highest macro-F1 score on startup, so restart the API after running this notebook. These scores are a single stratified holdout estimate, not a guarantee of future performance.